# 📊 CO Attainment Calculator — Detailed Walkthrough
---
This notebook implements a **complete Course Outcome (CO) Attainment** calculation pipeline
following the NBA OBE (Outcome-Based Education) framework.

### What This Notebook Does
1. Reads an Excel file containing student marks, question-to-CO mappings, and OBE configuration
2. Computes per-student CO percentages **per exam**, correctly handling unattempted questions
3. Aggregates across exams using **dynamic weightages** (Internal / External / Assignment)
4. Determines **class-level attainment levels** (0–3) based on configurable thresholds
5. Exports a clean Excel report with 3 sheets

### Key Features
| Feature | Description |
|---|---|
| Zero dependencies | Uses only Python standard library (`zipfile`, `xml`) |
| Dynamic CO count | Handles 2 to 8+ COs per exam automatically |
| Dynamic exam types | ST1, ST2, ST3, ETE, ASN, or any combination |
| Dynamic weightages | Reads Internal/External/Assignment splits from the data |
| Unattempted handling | `"U"`, blank, or `"AB"` excluded from both numerator & denominator |

### How to Use
1. Set `INPUT_FILES` in **Section 2** below
2. Click **Runtime → Run All** (Colab) or **Kernel → Restart & Run All** (Jupyter)
3. Find your output files as `Output_<name>.xlsx`

## Section 1: Lightweight XLSX Reader (No External Libraries)

### Why build our own reader?
Libraries like `openpyxl` or `pandas` require installation. To make this notebook
work anywhere (especially Google Colab with no setup), we read `.xlsx` files directly.

### How `.xlsx` files work internally
An `.xlsx` file is actually a **ZIP archive** containing XML files:

```
MyFile.xlsx (ZIP)
├── xl/
│   ├── sharedStrings.xml   ← All unique text values (strings)
│   ├── workbook.xml        ← Sheet names and ordering
│   └── worksheets/
│       ├── sheet1.xml      ← Raw cell data for Sheet 1
│       ├── sheet2.xml      ← Raw cell data for Sheet 2
│       └── ...
└── [Content_Types].xml     ← File type metadata
```

### How our reader works
1. **Open the ZIP** using Python's built-in `zipfile` module
2. **Parse `sharedStrings.xml`** — Excel stores text values in a shared pool to save space.
   Each cell just stores an index into this pool.
3. **Parse `workbook.xml`** — Get the ordered list of sheet names
4. **Parse each `sheet.xml`** — Each cell has a reference like `"B3"` (column B, row 3),
   a type (`t="s"` means shared string), and a value `<v>`.

### Key detail: Column gap handling
Excel only stores cells that have data. If column A and C have values but B is empty,
we need to fill that gap. The `_col_index()` method converts references like `"AB12"`
to numeric indices to handle this correctly.

In [ ]:
import os, re, sys, zipfile
import xml.etree.ElementTree as ET
from collections import OrderedDict

class XLSXReader:
    """
    Pure-Python .xlsx reader using zipfile + XML parsing.
    No external dependencies required (no pandas, no openpyxl).
    """

    def __init__(self, path):
        """
        Initialize the reader by loading metadata from the .xlsx file.
        
        Parameters:
            path (str): Path to the .xlsx file
        """
        self.path = path
        self.shared_strings = []       # List of all unique text values
        self.sheet_map = OrderedDict() # {sheet_name: xml_file_path}
        self._load_metadata()

    def _load_metadata(self):
        """
        Extract shared strings and sheet names from the ZIP archive.
        This is called once during initialization.
        """
        with zipfile.ZipFile(self.path) as z:
            # ── Step 1: Load shared strings ──
            # Excel stores all unique text in a pool to avoid duplication.
            # For example, if "ANSH GARG" appears in 5 cells, it's stored
            # once here, and each cell just references index 0, 1, etc.
            if 'xl/sharedStrings.xml' in z.namelist():
                tree = ET.fromstring(z.read('xl/sharedStrings.xml'))
                ns = {'m': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
                for si in tree.findall('m:si', ns):
                    # Some strings are split across multiple <t> elements
                    # (rich text formatting), so we join them all
                    texts = [t.text for t in si.findall('.//m:t', ns) if t.text is not None]
                    self.shared_strings.append("".join(texts))

            # ── Step 2: Load sheet names ──
            # The workbook.xml file lists all sheets in order
            wb = ET.fromstring(z.read('xl/workbook.xml'))
            ns = {'m': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
            for i, sheet in enumerate(wb.find('m:sheets', ns).findall('m:sheet', ns)):
                # Sheets are stored as sheet1.xml, sheet2.xml, etc.
                xml_path = f'xl/worksheets/sheet{i+1}.xml'
                if xml_path in z.namelist():
                    self.sheet_map[sheet.get('name')] = xml_path

    @property
    def sheet_names(self):
        """Return list of sheet names in workbook order."""
        return list(self.sheet_map.keys())

    def _col_index(self, ref):
        """
        Convert an Excel cell reference to a 0-based column index.
        
        Examples:
            'A1'  → 0    (column A)
            'B5'  → 1    (column B)
            'AA3' → 26   (column AA)
            'AB1' → 27   (column AB)
        
        The algorithm treats column letters as base-26 numbers:
            A=1, B=2, ..., Z=26, AA=27, AB=28, ...
        """
        col = ''
        for ch in ref:
            if ch.isalpha():
                col += ch
            else:
                break  # Stop at the first digit (row number)
        # Convert letters to number (A=1, B=2, ..., Z=26, AA=27)
        idx = 0
        for c in col.upper():
            idx = idx * 26 + (ord(c) - ord('A') + 1)
        return idx - 1  # Convert to 0-based

    def read_sheet(self, sheet_name):
        """
        Read all data from a sheet, returning a list of lists.
        
        Each inner list represents one row. Empty cells between
        data cells are filled with empty strings to maintain
        proper column alignment.
        
        Parameters:
            sheet_name (str): Name of the sheet to read
            
        Returns:
            list[list[str]]: 2D list of cell values as strings
        """
        ns = {'m': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
        with zipfile.ZipFile(self.path) as z:
            tree = ET.fromstring(z.read(self.sheet_map[sheet_name]))
        
        sd = tree.find('m:sheetData', ns)  # <sheetData> contains all rows
        rows = []
        
        for row_el in sd.findall('m:row', ns):
            row = []
            for c in row_el.findall('m:c', ns):
                ref = c.get('r', '')           # Cell reference like "B3"
                col_idx = self._col_index(ref)  # Convert to 0-based index
                
                # Fill gaps: if we're at column 5 but only have 3 values,
                # insert empty strings for columns 3 and 4
                while len(row) < col_idx:
                    row.append('')
                
                # Extract cell value
                v = c.find('m:v', ns)
                if v is not None and v.text is not None:
                    if c.get('t') == 's':
                        # Type 's' = shared string → look up in our pool
                        row.append(self.shared_strings[int(v.text)])
                    else:
                        # Numeric or other value → use as-is
                        row.append(v.text)
                else:
                    row.append('')  # Empty cell
            rows.append(row)
        
        return rows

# Quick test
print("✅ XLSXReader class loaded successfully.")
print("   Methods: __init__(), read_sheet(), sheet_names")

## Section 2: Configuration — Set Your Input Files

Edit the `INPUT_FILES` list below to point to your Excel files.

**Important notes:**
- Each file is processed **independently** (different files can have different CO counts, 
  different exams, and different weight schemes)
- Paths are relative to the notebook's working directory
- **Google Colab users**: Upload files first, then use paths like `"/content/Input 1.xlsx"`

In [ ]:
# ════════════════════════════════════════════════════════════
# CONFIGURATION: Set your input files here
# ════════════════════════════════════════════════════════════

INPUT_FILES = [
    "Input 1.xlsx",   # Each file processed independently
    "Input 2.xlsx",   
    "Input 3.xlsx",   
]

# Verify files exist before processing
for f in INPUT_FILES:
    status = "✅ Found" if os.path.exists(f) else "❌ NOT FOUND"
    print(f"  {status}: {f}")

## Section 3: Parse OBE Details (Thresholds & Weights)

The **first sheet** of each input file (`OBE Details`) contains the configuration
that controls how attainment is calculated. Here's what it looks like:

### Expected Layout
| Column A | Column B |
|---|---|
| **Threshold** | `60` |
| *(empty rows)* | |
| **Types** | **Weightages** |
| Internal (Avg of ST1,ST2,ST3) | `0.4` |
| External(ETE) | `0.6` |
| Assignment | `0.1` *(optional — only in some files)* |
| *(empty rows)* | |
| **CO Score** | **Percentage of students attaining Target** |
| 3 | `0.8` |
| 2 | `0.7` |
| 1 | `0.6` |

### What gets extracted
1. **`target_pct`** — The minimum CO% a student needs to "pass" a CO (e.g., 60%)
2. **`weights`** — How to combine Internal, External, and Assignment scores
3. **`levels`** — Thresholds for assigning attainment levels 1, 2, or 3

### How parsing works
The function uses a **state machine** approach:
- It scans rows top to bottom
- When it sees `"Types" | "Weightages"`, it switches to **weight-parsing mode**
- When it sees `"CO Score" | "...students..."`, it switches to **level-parsing mode**
- Category keywords (`internal`, `external`, `assign`) are matched case-insensitively

In [ ]:
def parse_obe_details(reader):
    """
    Parse the OBE Details sheet to extract configuration parameters.
    
    Parameters:
        reader (XLSXReader): An initialized reader for the input file
    
    Returns:
        tuple: (target_pct, weights, levels)
            - target_pct (float): Course target percentage (e.g., 60.0)
            - weights (OrderedDict): Category → weight mapping
              Example: {'Internal': 0.4, 'External': 0.6}
              or:      {'Internal': 0.3, 'External': 0.6, 'Assignment': 0.1}
            - levels (list): [(level_num, threshold), ...] sorted descending
              Example: [(3, 0.8), (2, 0.7), (1, 0.6)]
    """
    # Read all rows from the first sheet (OBE Details)
    rows = reader.read_sheet(reader.sheet_names[0])
    
    # Initialize with safe defaults (used if parsing fails)
    target_pct = 60.0
    weights = OrderedDict()
    levels = []
    
    # State machine: tracks which section we're currently parsing
    # None = looking for a section header
    # 'weights' = parsing weight rows
    # 'levels' = parsing level rows
    mode = None

    for row in rows:
        if len(row) < 2:
            continue  # Skip rows with less than 2 columns
        
        c0 = str(row[0]).strip()  # Column A
        c1 = str(row[1]).strip()  # Column B

        # ── Look for the Threshold row ──
        # This can appear anywhere, format: "Threshold | 60"
        if c0.lower() == 'threshold' and c1:
            try:
                target_pct = float(c1)
            except ValueError:
                pass  # Keep default if parsing fails
            continue

        # ── Detect section headers ──
        # When we see "Types | Weightages", switch to weight parsing
        if c0.lower() == 'types' and c1.lower() == 'weightages':
            mode = 'weights'
            continue
        
        # When we see "CO Score | ...students...", switch to level parsing
        if 'co score' in c0.lower() and 'student' in c1.lower():
            mode = 'levels'
            continue

        # ── Parse weight rows ──
        # Format: "Internal (Avg of ST1,ST2,ST3) | 0.4"
        if mode == 'weights' and c0 and c1:
            try:
                w = float(c1)
            except ValueError:
                continue  # Not a numeric value, skip
            
            # Categorise by matching keywords in column A
            c0_lower = c0.lower()
            if 'internal' in c0_lower or 'avg' in c0_lower:
                weights['Internal'] = w
            elif 'external' in c0_lower or 'ete' in c0_lower:
                weights['External'] = w
            elif 'assign' in c0_lower or 'asn' in c0_lower:
                weights['Assignment'] = w
            else:
                weights[c0] = w  # Unknown category, use original name

        # ── Parse level rows ──
        # Format: "3 | 0.8" meaning Level 3 requires 80% of students
        if mode == 'levels' and c0 and c1:
            try:
                lv = int(float(c0))  # Level number (1, 2, or 3)
                th = float(c1)       # Threshold (0.6, 0.7, or 0.8)
                levels.append((lv, th))
            except ValueError:
                pass

    # Sort levels descending so we match the HIGHEST level first
    # When checking: if rate >= 0.8 → Level 3, elif >= 0.7 → Level 2, etc.
    levels.sort(key=lambda x: x[0], reverse=True)

    # Apply defaults if nothing was found
    if not weights:
        weights = OrderedDict([('Internal', 0.4), ('External', 0.6)])
    if not levels:
        levels = [(3, 0.8), (2, 0.7), (1, 0.6)]

    return target_pct, weights, levels

print("✅ parse_obe_details() loaded.")
print("   Returns: (target_pct, weights_dict, levels_list)")

## Section 4: Discover Exams Automatically

Rather than hardcoding exam names like "ST1", "ST2", "ETE", the script **automatically
discovers** which exams exist by scanning the sheet names.

### Discovery logic
For each sheet name containing "Mapping":
1. Extract the exam name (everything before " Ques Mapping" or " Mapping")
2. Find a matching "Result" sheet (e.g., `ST1 Ques Mapping` → `ST1 Result`)
3. **Categorise** the exam based on keywords in its name:

| Name contains | Category | Examples |
|---|---|---|
| `ETE`, `External` | **External** | ETE, End Term Exam |
| `ASN`, `Assign` | **Assignment** | ASN, Assignment |
| Everything else | **Internal** | ST1, ST2, ST3, FA1 |

### Why categorisation matters
The OBE Details sheet specifies **weights per category**, not per exam. For example:
- Internal weight = 0.4 applies to the **average** of ST1, ST2, ST3
- External weight = 0.6 applies to ETE
- Assignment weight = 0.1 applies to ASN

In [ ]:
def discover_exams(reader):
    """
    Scan sheet names to find Mapping+Result pairs and categorise them.
    
    Parameters:
        reader (XLSXReader): An initialized reader
    
    Returns:
        list[dict]: Each dict has keys:
            - 'name': Exam name (e.g., 'ST1')
            - 'map_sheet': Full sheet name for mapping (e.g., 'ST1 Ques Mapping')
            - 'res_sheet': Full sheet name for results (e.g., 'ST1 Result')
            - 'category': 'Internal', 'External', or 'Assignment'
    """
    sheets = reader.sheet_names
    exams = []
    seen = set()  # Avoid duplicates
    
    for s in sheets:
        m = None
        if 'Mapping' in s:
            # Extract exam name by removing the "Mapping" suffix
            # "ST1 Ques Mapping" → "ST1"
            # "ASN Mapping" → "ASN"
            m = s.replace(' Ques Mapping', '').replace(' Mapping', '').strip()
        
        if m and m not in seen:
            # Find the corresponding Result sheet
            # Look for any sheet that starts with the exam name and contains "Result"
            res = next((s2 for s2 in sheets if s2.startswith(m) and 'Result' in s2), None)
            
            if res:
                # Categorise based on exam name keywords
                m_upper = m.upper()
                if m_upper.startswith('ETE') or 'EXTERNAL' in m_upper:
                    cat = 'External'
                elif m_upper.startswith('ASN') or 'ASSIGN' in m_upper:
                    cat = 'Assignment'
                else:
                    cat = 'Internal'  # ST1, ST2, ST3, FA, Quiz, etc.
                
                exams.append({
                    'name': m,
                    'map_sheet': s,
                    'res_sheet': res,
                    'category': cat
                })
                seen.add(m)
    
    return exams

print("✅ discover_exams() loaded.")
print("   Returns: list of {name, map_sheet, res_sheet, category}")

## Section 5: Parse Question Mappings & Student Results

### 5a. Mapping Sheet Structure

Each exam has a "Ques Mapping" sheet that defines **which questions test which COs**:

| Q_Id | Max Marks | CO1 | CO2 | CO3 | CO4 |
|---|---|---|---|---|---|
| Q1 | 5 | **1** | 0 | 0 | **1** |
| Q2 | 2 | 0 | **1** | 0 | 0 |
| Q3 | 10 | **1** | **1** | 0 | 0 |

**Reading this table:**
- Q1 (5 marks) tests **CO1** and **CO4** (value = 1 means mapped)
- Q2 (2 marks) tests only **CO2**
- Q3 (10 marks) tests **CO1** and **CO2** (many-to-many mapping)

**Important:** Different exams can have **different CO columns**. For example:
- ST1 might only have CO1, CO2 columns
- ST2 might only have CO3, CO4 columns
- ETE has all CO1–CO6

The parser detects CO columns dynamically using regex: any column header matching `CO` + digits.

### 5b. Result Sheet Structure

| Sr.no | Admission No. | Name | Course | Exam | Q1 | Q2 | Q3 | Total | Max |
|---|---|---|---|---|---|---|---|---|---|
| 1 | 2010990089 | ANSH | EC114 | ST1 | 4 | **U** | 8 | 12 | 17 |
| 2 | 2010992002 | PANSY | EC114 | ST1 | 5 | 2 | | 7 | 17 |

**Unattempted indicators** (all treated the same):
- `"U"` — explicitly marked unattempted
- `""` (empty/blank) — no entry
- `"AB"` — absent

These are stored as `None` in our data structure and **excluded from both the
numerator AND denominator** during CO% calculation.

### 5c. How we identify question columns
We skip any column whose header contains metadata keywords like `"admission"`, `"name"`,
`"course"`, `"exam"`, `"total"`, `"max"`. Everything else is treated as a question column.

In [ ]:
def parse_mapping(reader, sheet_name):
    """
    Parse a question mapping sheet to extract question-to-CO mappings.
    
    Parameters:
        reader (XLSXReader): Initialized reader
        sheet_name (str): Name of the mapping sheet
    
    Returns:
        tuple: (questions, all_cos)
            - questions: list of dicts, each with:
                'q_id' (str): Question identifier (e.g., 'Q1', '1a')
                'max_marks' (float): Maximum marks for this question
                'cos' (dict): {CO_name: 0_or_1} mapping
            - all_cos: sorted list of all CO names found in header
    
    Note: Questions with max_marks <= 0 or no CO mappings are excluded.
    This handles the "empty ST3" case where all values are 0.
    """
    rows = reader.read_sheet(sheet_name)
    if not rows:
        return [], []
    
    header = [str(h).strip() for h in rows[0]]
    
    # Find CO columns using regex: match 'CO1', 'CO2', ..., 'CO99'
    # re.IGNORECASE handles 'co1' or 'Co1' variants
    co_cols = [(i, h.upper()) for i, h in enumerate(header)
               if re.match(r'^CO\d+$', h, re.IGNORECASE)]
    
    questions = []
    for row in rows[1:]:  # Skip header row
        if len(row) < 2:
            continue
        
        # Column 0 = Question ID, Column 1 = Max Marks
        q_id = str(row[0]).strip()
        if not q_id:
            continue  # Skip empty rows
        
        try:
            max_m = float(row[1])
        except (ValueError, TypeError):
            continue  # Skip if max marks isn't numeric
        
        if max_m <= 0:
            continue  # Skip questions with 0 max marks (empty mappings)
        
        # Build CO mapping dict for this question
        cos = {}
        mapped_any = False
        for ci, cname in co_cols:
            try:
                val = int(float(row[ci])) if ci < len(row) else 0
            except (ValueError, TypeError):
                val = 0
            cos[cname] = val
            if val > 0:
                mapped_any = True
        
        # Only include if at least one CO is mapped
        if mapped_any:
            questions.append({'q_id': q_id, 'max_marks': max_m, 'cos': cos})
    
    all_cos = sorted(set(c for _, c in co_cols))
    return questions, all_cos


def parse_results(reader, sheet_name):
    """
    Parse a student results sheet.
    
    Parameters:
        reader (XLSXReader): Initialized reader
        sheet_name (str): Name of the results sheet
    
    Returns:
        list[dict]: Each dict has:
            'roll' (str): Student roll/admission number
            'name' (str): Student name
            'marks' (dict): {question_id: float_or_None}
                float = attempted marks, None = unattempted
    """
    rows = reader.read_sheet(sheet_name)
    if not rows:
        return []
    
    header = [str(h).strip() for h in rows[0]]
    
    # ── Find roll number and name columns ──
    roll_idx = name_idx = None
    for i, h in enumerate(header):
        hl = h.lower()
        if 'admission' in hl or 'roll' in hl:
            roll_idx = i
        if 'name' in hl and 'student' in hl:
            name_idx = i
    
    if roll_idx is None:
        return []  # Can't identify students without roll numbers
    
    # ── Identify question columns ──
    # Strategy: exclude columns with metadata keywords, keep the rest
    meta_kw = {'sr.no', 'sr no', 'admission', 'roll', 'name', 'course',
               'exam', 'total', 'max', 'maximum'}
    q_cols = []
    for i, h in enumerate(header):
        hl = h.lower().strip()
        is_meta = any(kw in hl for kw in meta_kw)
        if not is_meta and h.strip():
            q_cols.append((i, h.strip()))
    
    # ── Parse each student row ──
    students = []
    for row in rows[1:]:
        if len(row) <= max(roll_idx, 1):
            continue
        
        # Clean roll number (remove non-breaking spaces \xa0)
        roll = str(row[roll_idx]).strip().replace('\xa0', '').strip()
        if not roll:
            continue
        
        # Clean student name
        name = ''
        if name_idx is not None and name_idx < len(row):
            name = str(row[name_idx]).strip().replace('\xa0', '').strip()
        
        # ── Parse marks for each question ──
        marks = {}
        for ci, qname in q_cols:
            if ci < len(row):
                val = str(row[ci]).strip().upper()
                if val in ('U', '', 'AB'):
                    marks[qname] = None  # Unattempted
                else:
                    try:
                        marks[qname] = float(val)
                    except (ValueError, TypeError):
                        marks[qname] = None  # Can't parse → treat as unattempted
            else:
                marks[qname] = None  # Column doesn't exist for this row
        
        students.append({'roll': roll, 'name': name, 'marks': marks})
    
    return students

print("✅ parse_mapping() and parse_results() loaded.")

## Section 6: Core Calculation — Per-Student CO Percentage

This is the **most important function** in the entire notebook. It implements the
fundamental CO attainment formula from the NBA OBE framework.

### The Formula

```
              Σ (obtained marks on attempted questions mapped to this CO)
CO % = ──────────────────────────────────────────────────────────────────── × 100
              Σ (max marks on attempted questions mapped to this CO)
```

### The Critical Unattempted Rule

When a question is marked as **unattempted** (`"U"`, blank, or `"AB"`):
- Its marks are **NOT** added to the numerator (obviously — the student scored 0)
- Its max marks are **also NOT** added to the denominator

**Why?** This prevents unfair penalisation. Consider:

| Scenario | Q1 (10 max) | Q2 (10 max) | Naive CO% | Correct CO% |
|---|---|---|---|---|
| Both attempted | 8 | 6 | 14/20 = 70% | 14/20 = **70%** |
| Q2 unattempted | 8 | U | 8/20 = 40% ❌ | 8/10 = **80%** ✅ |

The naive approach (keeping 20 in denominator) would give 40%, but the
student actually scored 80% on what they attempted.

### Edge Cases
- If a student didn't attempt **any** question for a CO → returns `None` (not 0%)
- If a CO has no mapped questions in this exam → returns `None`

In [ ]:
def calc_student_co_pct(student_marks, questions, co_name):
    """
    Calculate the CO percentage for one student in one exam.
    
    This function implements the core NBA OBE attainment formula:
    CO% = (sum of obtained marks) / (sum of max marks) × 100
    where ONLY attempted questions mapped to this CO are included.
    
    Parameters:
        student_marks (dict): {question_id: float_or_None}
            float = marks obtained, None = unattempted
        questions (list): List of question dicts from parse_mapping()
        co_name (str): The CO to calculate for (e.g., 'CO1')
    
    Returns:
        float or None: Percentage (0-100), or None if no valid data
    
    Example:
        >>> marks = {'Q1': 8, 'Q2': None, 'Q3': 6}
        >>> questions = [
        ...     {'q_id': 'Q1', 'max_marks': 10, 'cos': {'CO1': 1}},
        ...     {'q_id': 'Q2', 'max_marks': 10, 'cos': {'CO1': 1}},
        ...     {'q_id': 'Q3', 'max_marks': 5,  'cos': {'CO2': 1}},
        ... ]
        >>> calc_student_co_pct(marks, questions, 'CO1')
        80.0  # = 8/10 × 100 (Q2 excluded, Q3 not mapped to CO1)
    """
    obtained = 0.0   # Running total of marks earned
    max_total = 0.0  # Running total of maximum possible marks
    
    for q in questions:
        # ── Step 1: Check if this question is mapped to the CO ──
        # cos dict has {CO_name: 0_or_1}. Skip if not mapped (0).
        if q['cos'].get(co_name, 0) <= 0:
            continue  # This question doesn't test this CO
        
        # ── Step 2: Check if the student attempted this question ──
        val = student_marks.get(q['q_id'])
        if val is None:
            # CRITICAL: Skip unattempted questions in BOTH
            # numerator AND denominator. This is the key rule
            # from the NBA OBE framework (PRD 11, Section 4.1).
            continue
        
        # ── Step 3: Accumulate marks ──
        obtained += val           # Add what the student scored
        max_total += q['max_marks']  # Add the maximum possible
    
    # ── Step 4: Calculate percentage ──
    if max_total == 0:
        return None  # No valid questions → can't calculate
    
    return (obtained / max_total) * 100.0

print("✅ calc_student_co_pct() loaded.")
print("   Core formula: CO% = Σ(obtained) / Σ(max_attempted) × 100")

## Section 7: Full Processing Pipeline

This function ties everything together for **one input file**:

### Processing Flow
```
Step 1: Parse OBE Config (target, weights, levels)
    ↓
Step 2: Discover exams (ST1, ST2, ETE, ASN...)
    ↓
Step 3: Parse mappings + results for each exam
    ↓
Step 4: Calculate per-student CO% for every exam
    ↓
Step 5: Compute weighted average across categories
    ↓
Step 6: Check each student against target threshold
    ↓
Step 7: Calculate class success rate per CO
    ↓
Step 8: Assign attainment level (0, 1, 2, or 3)
```

### Step 5 Detail: Weighted Average
The formula combines Internal, External, and (optionally) Assignment scores:

```
Final CO% = (Internal_Avg × W_int) + (External% × W_ext) + (Assignment% × W_asn)
```

Where `Internal_Avg` is the **mean** of all Internal exam CO percentages for that student.
For example, if ST1_CO1=80% and ST2_CO1=60%, then Internal_Avg_CO1 = 70%.

### Step 7–8 Detail: Class Aggregation
```
Success Rate = (# students with Final CO% ≥ Target) / (# students with valid data) × 100

Level 3 if Success Rate ≥ 80%
Level 2 if Success Rate ≥ 70%
Level 1 if Success Rate ≥ 60%
Level 0 otherwise
```

In [ ]:
def process_file(filepath):
    """
    End-to-end processing of one input Excel file.
    
    Parameters:
        filepath (str): Path to the input .xlsx file
    
    Returns:
        tuple: (all_students, student_final_co, attainment, meta)
    """
    print(f"\n{'='*60}")
    print(f"  Processing: {os.path.basename(filepath)}")
    print(f"{'='*60}")

    reader = XLSXReader(filepath)

    # ════ Step 1: Parse OBE configuration ════
    target_pct, weights, levels = parse_obe_details(reader)
    print(f"  Target: {target_pct}%")
    print(f"  Weights: {dict(weights)}")
    print(f"  Levels: {levels}")

    # ════ Step 2: Discover exams ════
    exams = discover_exams(reader)
    print(f"  Exams: {[e['name']+' ('+e['category']+')' for e in exams]}")

    # ════ Step 3: Parse mapping + results for each exam ════
    exam_data = []
    all_cos_global = set()
    all_students = OrderedDict()  # roll → name (preserves insertion order)

    for exam in exams:
        questions, cos = parse_mapping(reader, exam['map_sheet'])
        students = parse_results(reader, exam['res_sheet'])
        
        if not questions:
            # This handles "empty" exams like ST3 where all max marks = 0
            print(f"    ⏭️  Skipping {exam['name']}: no valid question mappings")
            continue
        
        # Collect all unique students across all exams
        for s in students:
            if s['roll'] not in all_students:
                all_students[s['roll']] = s['name']
        
        all_cos_global.update(cos)
        exam_data.append({
            'name': exam['name'], 'category': exam['category'],
            'questions': questions, 'cos': cos,
            'students': {s['roll']: s for s in students}
        })

    all_cos = sorted(all_cos_global)
    print(f"  COs detected: {all_cos}")
    print(f"  Total students: {len(all_students)}")

    # ════ Step 4: Per-student, per-exam CO percentages ════
    # Build a 3D structure: student_exam_co[roll][exam_name][CO] = pct
    student_exam_co = {}
    for roll in all_students:
        student_exam_co[roll] = {}
        for ed in exam_data:
            s_data = ed['students'].get(roll)
            if s_data is None:
                # Student not in this exam's result sheet
                student_exam_co[roll][ed['name']] = {co: None for co in all_cos}
            else:
                student_exam_co[roll][ed['name']] = {
                    co: calc_student_co_pct(s_data['marks'], ed['questions'], co)
                    for co in all_cos
                }

    # ════ Step 5: Weighted final CO% per student ════
    # Group exams by their category for weighted averaging
    cat_exams = {}
    for ed in exam_data:
        cat_exams.setdefault(ed['category'], []).append(ed['name'])

    student_final_co = {}
    for roll in all_students:
        student_final_co[roll] = {}
        for co in all_cos:
            weighted_total = 0.0
            
            for cat, w in weights.items():
                cat_exam_names = cat_exams.get(cat, [])
                if not cat_exam_names:
                    continue
                
                # Collect CO% from all exams in this category
                pcts = []
                for ename in cat_exam_names:
                    p = student_exam_co[roll].get(ename, {}).get(co)
                    if p is not None:
                        pcts.append(p)
                
                # Average across exams in this category, then apply weight
                if pcts:
                    cat_avg = sum(pcts) / len(pcts)
                    weighted_total += cat_avg * w
            
            student_final_co[roll][co] = weighted_total if weighted_total > 0 else None

    # ════ Step 6-8: Class-level aggregation & attainment ════
    attainment = OrderedDict()
    for co in all_cos:
        valid = [(r, p) for r, d in student_final_co.items()
                 if (p := d.get(co)) is not None]
        total = len(valid)
        passed = sum(1 for _, p in valid if p >= target_pct)
        rate = passed / total if total > 0 else 0.0
        
        # Assign level: check from highest to lowest
        level = 0
        for lv, th in levels:
            if rate >= th:
                level = lv
                break
        
        attainment[co] = {
            'Attempted': total, 'Met Target': passed,
            'Success Rate %': round(rate * 100, 2), 'Level': level
        }

    # ════ Print summary table ════
    print(f"\n  {'CO':<8} {'Attempted':>10} {'Met Target':>12} {'Success%':>10} {'Level':>6}")
    print(f"  {'-'*50}")
    for co, info in attainment.items():
        print(f"  {co:<8} {info['Attempted']:>10} {info['Met Target']:>12} "
              f"{info['Success Rate %']:>9.2f}% {info['Level']:>6}")

    return all_students, student_final_co, attainment, {
        'target': target_pct, 'weights': dict(weights),
        'levels': levels, 'cos': all_cos
    }

print("✅ process_file() loaded — full pipeline ready.")

## Section 8: Excel Output Writer

Writes results to a `.xlsx` file with **3 sheets**:

| Sheet | Contents |
|---|---|
| **Student Details** | Roll No, Name, CO percentages, Target Met (Yes/No) per CO |
| **Attainment Summary** | Per-CO: students attempted, met target, success rate, level |
| **Configuration** | Target %, weights, level thresholds used |

### How it works
Since we can't use `openpyxl`, we build the XLSX **from raw XML**:
1. Build a shared strings pool (all unique text values)
2. Generate XML for each sheet's rows and cells
3. Package everything into a ZIP file with the correct structure

In [ ]:
def write_output_xlsx(filepath, all_students, student_final_co, attainment, meta):
    """Write results to a .xlsx file using only zipfile + XML."""
    all_cos = meta['cos']
    ss = []; ss_map = {}

    def add_ss(s):
        s = str(s)
        if s not in ss_map:
            ss_map[s] = len(ss); ss.append(s)
        return ss_map[s]

    def col_letter(idx):
        result = ''
        while True:
            result = chr(idx % 26 + ord('A')) + result
            idx = idx // 26 - 1
            if idx < 0: break
        return result

    def make_row(rn, cells_data):
        parts = [f'<row r="{rn}">']
        for ci, (val, is_s) in enumerate(cells_data):
            ref = f'{col_letter(ci)}{rn}'
            if is_s:
                parts.append(f'<c r="{ref}" t="s"><v>{add_ss(str(val))}</v></c>')
            else:
                parts.append(f'<c r="{ref}"><v>{val}</v></c>')
        parts.append('</row>')
        return ''.join(parts)

    ns = 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'
    rns = 'http://schemas.openxmlformats.org/officeDocument/2006/relationships'
    pns = 'http://schemas.openxmlformats.org/package/2006/relationships'
    cns = 'http://schemas.openxmlformats.org/package/2006/content-types'

    # ── Sheet 1: Student Details ──
    s1 = []
    hdrs = ['Roll No.','Name'] + [f'{c} %' for c in all_cos] + [f'{c} Target' for c in all_cos]
    s1.append(make_row(1, [(h, True) for h in hdrs]))
    rn = 2
    for roll, name in all_students.items():
        cs = [(roll, True), (name, True)]
        for co in all_cos:
            p = student_final_co[roll].get(co)
            cs.append((round(p, 2), False) if p else ('N/A', True))
        for co in all_cos:
            p = student_final_co[roll].get(co)
            cs.append(('Yes' if p and p >= meta['target'] else ('No' if p else 'N/A'), True))
        s1.append(make_row(rn, cs)); rn += 1

    # ── Sheet 2: Attainment Summary ──
    s2 = []
    s2.append(make_row(1, [(h, True) for h in ['CO','Attempted','Met Target','Success %','Level']]))
    rn = 2
    for co, info in attainment.items():
        s2.append(make_row(rn, [(co,True),(info['Attempted'],False),(info['Met Target'],False),(info['Success Rate %'],False),(info['Level'],False)])); rn += 1

    # ── Sheet 3: Configuration ──
    s3 = []
    s3.append(make_row(1, [('Parameter',True),('Value',True)]))
    rn = 2
    for k,v in [('Target %',meta['target'])]+[(f'Weight: {c}',w) for c,w in meta['weights'].items()]+[(f'Level {l} Threshold',t) for l,t in meta['levels']]:
        s3.append(make_row(rn, [(str(k),True),(v,False)])); rn += 1

    def sxml(rows):
        return f'<?xml version="1.0" encoding="UTF-8"?><worksheet xmlns="{ns}"><sheetData>{"".join(rows)}</sheetData></worksheet>'

    ss_xml = f'<?xml version="1.0" encoding="UTF-8"?><sst xmlns="{ns}" count="{len(ss)}" uniqueCount="{len(ss)}">'
    for s in ss:
        ss_xml += f'<si><t>{s.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")}</t></si>'
    ss_xml += '</sst>'

    with zipfile.ZipFile(filepath, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.writestr('[Content_Types].xml', f'<?xml version="1.0"?><Types xmlns="{cns}"><Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/><Default Extension="xml" ContentType="application/xml"/><Override PartName="/xl/workbook.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/><Override PartName="/xl/worksheets/sheet1.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/><Override PartName="/xl/worksheets/sheet2.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/><Override PartName="/xl/worksheets/sheet3.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/><Override PartName="/xl/sharedStrings.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sharedStrings+xml"/></Types>')
        zf.writestr('_rels/.rels', f'<?xml version="1.0"?><Relationships xmlns="{pns}"><Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" Target="xl/workbook.xml"/></Relationships>')
        zf.writestr('xl/workbook.xml', f'<?xml version="1.0"?><workbook xmlns="{ns}" xmlns:r="{rns}"><sheets><sheet name="Student Details" sheetId="1" r:id="rId1"/><sheet name="Attainment Summary" sheetId="2" r:id="rId2"/><sheet name="Configuration" sheetId="3" r:id="rId3"/></sheets></workbook>')
        zf.writestr('xl/_rels/workbook.xml.rels', f'<?xml version="1.0"?><Relationships xmlns="{pns}"><Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet1.xml"/><Relationship Id="rId2" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet2.xml"/><Relationship Id="rId3" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet3.xml"/><Relationship Id="rId4" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/sharedStrings" Target="sharedStrings.xml"/></Relationships>')
        zf.writestr('xl/sharedStrings.xml', ss_xml)
        zf.writestr('xl/worksheets/sheet1.xml', sxml(s1))
        zf.writestr('xl/worksheets/sheet2.xml', sxml(s2))
        zf.writestr('xl/worksheets/sheet3.xml', sxml(s3))
    print(f"  Saved: {filepath}")

print("✅ write_output_xlsx() loaded.")

## Section 9: Execute — Process All Input Files

The cell below runs the full pipeline on each file in `INPUT_FILES`.
Each file is processed **independently** (different exams, COs, and weights are fine).

### Output files
For each `Input X.xlsx`, an `Output_Input X.xlsx` is created with:
- **Student Details** sheet — every student's final CO% and pass/fail
- **Attainment Summary** sheet — class-level results
- **Configuration** sheet — the parameters used

In [ ]:
for filepath in INPUT_FILES:
    if not os.path.exists(filepath):
        print(f"  File not found: {filepath}")
        continue
    try:
        students, final_co, attainment, meta = process_file(filepath)
        base = os.path.splitext(os.path.basename(filepath))[0]
        out = os.path.join(os.path.dirname(filepath) or '.', f"Output_{base}.xlsx")
        write_output_xlsx(out, students, final_co, attainment, meta)
    except Exception as e:
        print(f"  Error processing {filepath}: {e}")
        import traceback; traceback.print_exc()

print(f"\n{'='*60}")
print("  All files processed!")
print(f"{'='*60}")

## Appendix: CO Attainment Theory Reference (NBA OBE)

### The Conceptual Framework
In Outcome-Based Education (OBE), we don't just care about a student's total score.
We measure their proficiency in specific **Course Outcomes (COs)**.

### Complete Process Chain

| Step | Action | Level |
|---|---|---|
| 1 | Define COs for the course (CO1, CO2, ...) | Course setup |
| 2 | Map exam questions to COs (many-to-many) | Assessment design |
| 3 | Record student marks per question | Data collection |
| 4 | Calculate per-student CO% (excluding unattempted) | **Individual** |
| 5 | Apply weighted average (Internal + External + Assignment) | **Individual** |
| 6 | Check if student meets target (e.g., ≥ 60%) | **Individual** |
| 7 | Count % of class meeting target | **Class** |
| 8 | Assign attainment level (0–3) | **Class** |

### Data Integrity Rules
1. **Unattempted questions**: Both obtained (0) and max marks excluded from calculation
2. **Multiple CO mapping**: A question mapped to CO1 & CO2 contributes fully to BOTH
3. **Weightage by marks**: A 20-mark question has 2× impact vs a 10-mark question
4. **Category averaging**: Internal = average of ST1, ST2, ST3 (if present)

### Example Calculation
- Student scores 8/10 on Q1 (CO1), Q2 (CO1) is Unattempted
- CO1% = 8/10 = 80% (Q2 excluded from denominator)
- If target is 60%: Student **passes** CO1
- If 70% of the class passes: Attainment Level = **2** (assuming L2 threshold = 70%)